In [4]:
# ============================================================================
# ARQUIVO: notebooks/02_correlacoes_visuais.ipynb
# DESCRIÇÃO: Visualização Estatística (Heatmaps, Boxplots e Distribuições)
# ============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import pyspark.sql.functions as F

# Configuração estética global dos gráficos
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ----------------------------------------------------------------------------
# 1. PREPARAÇÃO DO DATASET ANALÍTICO (JOIN ENTRE FATOS E DIMENSÕES DA GOLD)
# ----------------------------------------------------------------------------
# Agrega o saldo médio e volume transacional por conta a partir de fato_transacoes
df_saldo_conta = (
    spark.table("portfolio.gold.fato_transacoes")
    .groupBy("id_conta")
    .agg(
        F.round(F.avg("saldo"), 2).alias("saldo_medio_conta"),
        F.round(F.avg("valor"), 2).alias("ticket_medio_transacao"),
        F.count("id_transacao").alias("qtd_transacoes")
    )
)

# Une com os empréstimos (fato_emprestimos), disposição (OWNER) e clientes (dim_cliente)
df_emprestimos = spark.table("portfolio.gold.fato_emprestimos")
df_disp = spark.table("portfolio.gold.dim_Disposicao").filter(F.upper(F.col("tipo_disposicao")).isin("OWNER", "TITULAR"))
df_clientes = spark.table("portfolio.gold.dim_cliente")
df_cartoes = spark.table("portfolio.gold.dim_cartao")

# Constrói o DataFrame consolidado em Pandas para plotagem estatística
df_visual_spark = (
    df_saldo_conta
    .join(df_emprestimos, on="id_conta", how="left")
    .join(df_disp, on="id_conta", how="left")
    .join(df_clientes, on="id_cliente", how="left")
    .join(df_cartoes, on="id_disp", how="left")
    .withColumn("possui_cartao", F.when(F.col("id_cartao").isNotNull(), "Com Cartão").otherwise("Sem Cartão"))
)

df_pd = df_visual_spark.toPandas()

# ----------------------------------------------------------------------------
# GRÁFICO 1: MATRIZ DE CORRELAÇÃO (HEATMAP) - VARIÁVEIS FINANCEIRAS E ETÁRIAS
# ----------------------------------------------------------------------------
cols_numericas = [
    "saldo_medio_conta",
    "ticket_medio_transacao",
    "qtd_transacoes",
    "valor_emprestimo",
    "duracao",
    "parcelas",
    "idade"
]
# Filtra apenas as colunas presentes no schema e calcula a correlação
cols_existentes = [c for c in cols_numericas if c in df_pd.columns]
matriz_corr = df_pd[cols_existentes].corr(method="spearman")

fig, ax = plt.subplots(figsize=(10, 8))
mascara = np.triu(np.ones_like(matriz_corr, dtype=bool))
sns.heatmap(
    matriz_corr,
    mask=mascara,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    linewidths=0.8,
    cbar_kws={"label": "Coeficiente de Correlação de Spearman"},
    ax=ax
)
ax.set_title("Matriz de Correlação: Comportamento Transacional vs. Crédito (Banco Berka)", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------------------------
# GRÁFICO 2: BOXPLOT - SALDO MÉDIO E VALOR DO EMPRÉSTIMO POR STATUS DE RISCO
# ----------------------------------------------------------------------------
df_loans_pd = df_pd.dropna(subset=["status_emprestimo", "valor_emprestimo"]).copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(
    data=df_loans_pd,
    x="status_emprestimo",
    y="valor_emprestimo",
    hue="status_emprestimo",
    palette="Set2",
    legend=False,
    ax=axes[0]
)
axes[0].set_title("Distribuição e Outliers de Valor do Empréstimo por Status", fontweight="bold")
axes[0].set_xlabel("Status do Contrato de Empréstimo")
axes[0].set_ylabel("Valor do Empréstimo (CZK)")
axes[0].tick_params(axis="x", rotation=20)

sns.boxplot(
    data=df_loans_pd,
    x="status_emprestimo",
    y="saldo_medio_conta",
    hue="status_emprestimo",
    palette="Set2",
    legend=False,
    ax=axes[1]
)
axes[1].set_title("Saldo Médio da Conta por Status de Adimplência", fontweight="bold")
axes[1].set_xlabel("Status do Contrato de Empréstimo")
axes[1].set_ylabel("Saldo Médio em Conta (CZK)")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

# ----------------------------------------------------------------------------
# GRÁFICO 3: HISTOGRAMA + KDE - PROVA DA HIPÓTESE DE ELEGIBILIDADE A CARTÃO
# Regra documentada no PDF: Saldo Médio <= 37.097,18 CZK não possui cartão
# ----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 6))

sns.histplot(
    data=df_pd,
    x="saldo_medio_conta",
    hue="possui_cartao",
    kde=True,
    bins=45,
    stat="density",
    common_norm=False,
    alpha=0.45,
    ax=ax
)

# Linha vertical demarcando a regra de negócio de 37.097,18 CZK do estudo PKDD'99
ax.axvline(37097.18, color="#d62728", linestyle="--", linewidth=2.2, label="Limiar PKDD'99 (37.097,18 CZK)")
ax.axvline(66899.77, color="#2ca02c", linestyle=":", linewidth=2.2, label="Alvo Praga Alta Renda (66.899,77 CZK)")

ax.set_title("Distribuição do Saldo Médio da Conta: Clientes Com vs. Sem Cartão de Crédito", fontsize=13, fontweight="bold")
ax.set_xlabel("Saldo Médio da Conta (CZK)")
ax.set_ylabel("Densidade")
ax.legend()
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'